### Chapter 6.5 - Custom Layers

Custom layers let you define reusable computations that fit into PyTorch models like built-in layers.


In [ ]:
from pathlib import Path
import tempfile
import torch


### 6.5.1 Layers without Parameters

#### 1. Intuition

A layer without parameters performs computation but has no learned weights or biases.

It still inherits from `torch.nn.Module` so it can be placed inside larger models.

#### 2. Why this exists

Some useful transformations do not need learned parameters, such as centering, reshaping, or clipping.


#### 3. Examples

Define a parameter-free layer.


In [ ]:
class CenteredLayer(torch.nn.Module):
    def forward(self, X):
        return X - X.mean()

layer = CenteredLayer()
Y = layer(torch.tensor([1.0, 2.0, 3.0]))
Y.mean()


#### 4. Step-by-step breakdown

The class inherits from `Module`.

It defines only `forward` because no parameters need initialization.

The output subtracts the mean from every input value.

The output mean is zero apart from floating-point rounding.

#### 5. Connection to ML systems

Parameter-free layers are common for deterministic transformations inside a model pipeline.

#### 6. Common confusion points

- A module does not need parameters.
- `forward` can use ordinary tensor operations.
- Parameter-free does not mean useless.
- The layer can still be part of `Sequential`.


### 6.5.2 Layers with Parameters

#### 1. Intuition

A layer with parameters stores learned tensors as `torch.nn.Parameter` objects.

`Parameter` tells PyTorch that a tensor should be treated as trainable model state.

#### 2. Why this exists

Custom parameterized layers are needed when built-in layers do not match the computation you want.


#### 3. Examples

Define a custom linear layer.


In [ ]:
class MyLinear(torch.nn.Module):
    def __init__(self, in_units, out_units):
        super().__init__()
        self.weight = torch.nn.Parameter(torch.randn(in_units, out_units) * 0.01)
        self.bias = torch.nn.Parameter(torch.zeros(out_units))
    def forward(self, X):
        return X @ self.weight + self.bias

layer = MyLinear(2, 3)
layer(torch.randn(4, 2)).shape


#### 4. Step-by-step breakdown

`Parameter` wraps tensors so PyTorch registers them as learnable.

`self.weight` and `self.bias` are stored as module attributes.

The forward method computes a linear transformation.

Calling the layer runs that forward computation.

#### 5. Connection to ML systems

Custom parameterized layers are central to research code and new architectures.

#### 6. Common confusion points

- Use `Parameter` for tensors that should be learned.
- Store parameters on `self` so the module can find them.
- Shape choices must match the forward computation.
- Custom layers should be tested with tiny inputs.


### 6.5.3 Summary

#### 1. Intuition

Custom layers can be parameter-free or parameterized.

Both forms use `forward` to define computation.

#### 2. Why this exists

Custom layers let you extend PyTorch while keeping compatibility with modules, optimizers, and model inspection tools.


#### 3. Examples

Inspect custom layer parameters.


In [ ]:
layer = MyLinear(2, 3)
[(name, p.shape) for name, p in layer.named_parameters()]


#### 4. Step-by-step breakdown

`named_parameters()` finds the custom layer's registered parameters.

The names come from the attributes `weight` and `bias`.

The shapes match the custom initialization.

#### 5. Connection to ML systems

If a custom layer registers parameters correctly, optimizers can update them like built-in layer parameters.

#### 6. Common confusion points

- Registration happens by assigning `Parameter` objects to `self`.
- Plain tensors on `self` are not automatically trainable parameters.
- Custom layers should expose predictable shapes.
- Test forward output before training.


### 6.5.4 Exercises

#### 1. Intuition

These exercises practice custom layer creation.

#### 2. Why this exists

Writing tiny layers makes module mechanics concrete.


#### 3. Examples

Exercise 1: create a layer that multiplies by two.


In [ ]:
class DoubleLayer(torch.nn.Module):
    def forward(self, X):
        return 2 * X

DoubleLayer()(torch.tensor([3.0]))


Exercise 2: list parameters of a parameter-free layer.


In [ ]:
list(DoubleLayer().parameters())


#### 4. Step-by-step breakdown

Exercise 1 defines a parameter-free computation.

Exercise 2 confirms there are no trainable parameters.

This distinction matters for optimizer behavior.

#### 5. Connection to ML systems

Custom layers are safe to add to larger models when their input/output behavior is clear.

#### 6. Common confusion points

- Parameter-free layers return an empty parameter list.
- Parameterized layers need `Parameter` objects.
- The forward method should be deterministic unless randomness is intended.
- Test with one small tensor first.
